# Práctica 2. Representación algebraica y geométrica de un qubit

**Tema:** Conceptos fundamentales. Superposición cuántica  
**Duración estimada:** 60–75 minutos  
**Modalidad:** Individual  
**Lenguaje:** Python  
**Bibliotecas:** NumPy y Matplotlib  
**Uso de Qiskit:** No se utilizará en esta práctica.

En esta práctica trabajaremos con números complejos, notación de Dirac, productos internos, bases ortonormales, fase global y esfera de Bloch.

## Objetivos

Al finalizar la práctica serás capaz de:

- Calcular conjugados, módulos y fases de números complejos.
- Diferenciar $z^2$ y $|z|^2$.
- Representar *kets* y *bras* con NumPy.
- Calcular productos internos y comprobar la ortogonalidad.
- Validar bases ortonormales.
- Distinguir fase global y fase relativa.

> Completa las celdas marcadas con `TODO` y responde en Markdown a todas las preguntas.

## Entrega

Entrega este notebook con el nombre:

```text
practica_02_representacion_qubit_apellidos_nombre.ipynb
```

Debe contener código ejecutable, resultados, gráficas, respuestas razonadas, pruebas y una conclusión final. No se considerará completa una entrega formada únicamente por código.

## Preparación del entorno

In [16]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)
TOLERANCIA = 1e-10


def es_estado_valido(estado, tolerancia=1e-10):
    try:
        vector = np.asarray(estado, dtype=complex)
    except (TypeError, ValueError):
        return False

    if vector.shape != (2,) or not np.all(np.isfinite(vector)):
        return False

    norma = np.linalg.norm(vector)
    return (
        not np.isclose(norma, 0.0, atol=tolerancia)
        and np.isclose(norma, 1.0, atol=tolerancia, rtol=0.0)
    )


def normalizar_estado(estado):
    vector = np.asarray(estado, dtype=complex)

    if vector.shape != (2,):
        raise ValueError("Un qubit debe tener dos amplitudes")
    if not np.all(np.isfinite(vector)):
        raise ValueError("Las amplitudes deben ser finitas")

    norma = np.linalg.norm(vector)
    if np.isclose(norma, 0.0):
        raise ValueError("El vector nulo no puede normalizarse")

    return vector / norma


ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)

# Parte A. Números complejos

## Ejercicio 1. Componentes de un número complejo

Considera:

$$z_1=3+4i,\qquad z_2=1-i,\qquad z_3=-2i,\qquad z_4=-1-\sqrt{3}i.$$

Para cada número muestra su parte real, parte imaginaria, conjugado, módulo, módulo al cuadrado, fase en radianes y fase en grados.

In [17]:
# TODO: representa los cuatro números complejos
z_1 = 3+4j
z_2 = 1-1j
z_3 = -2j
z_4 = -1-np.sqrt(3)*1j

numeros = {"z_1": z_1, "z_2": z_2, "z_3": z_3, "z_4": z_4}

# TODO: recorre el diccionario y muestra los datos solicitados

for nombre, z in numeros.items():
    print(
        f"{nombre} = {z:.5f} | Re={z.real:.5f} | Im={z.imag:.5f} |"
        f"conjugado={np.conj(z):.5f} | módulo={np.abs(z):.5f} | "
        f"fase={np.angle(z):.5f} rad({np.degrees(np.angle(z)):.2f}º)"
    )

z_1 = 3.00000+4.00000j | Re=3.00000 | Im=4.00000 |conjugado=3.00000-4.00000j | módulo=5.00000 | fase=0.92730 rad(53.13º)
z_2 = 1.00000-1.00000j | Re=1.00000 | Im=-1.00000 |conjugado=1.00000+1.00000j | módulo=1.41421 | fase=-0.78540 rad(-45.00º)
z_3 = -0.00000-2.00000j | Re=-0.00000 | Im=-2.00000 |conjugado=-0.00000+2.00000j | módulo=2.00000 | fase=-1.57080 rad(-90.00º)
z_4 = -1.00000-1.73205j | Re=-1.00000 | Im=-1.73205 |conjugado=-1.00000+1.73205j | módulo=2.00000 | fase=-2.09440 rad(-120.00º)


**Responde:**

1. ¿Puede el módulo ser negativo?
2. ¿Qué relación existe entre la fase de un número y la de su conjugado?
3. ¿En qué cuadrante se encuentra cada número?
4. ¿Por qué conviene utilizar `np.angle()` en lugar de calcular simplemente $\arctan(b/a)$?

**Respuesta:** 

- No, el módulo nunca es negativo.
- La fase del conjugado es la fase opuesta, salvo equivalencias módulo (2\pi).
- (z_1): primer cuadrante. (z_2): cuarto. (z_3): eje imaginario negativo. (z_4): tercer cuadrante.
- Porque np.angle() determina correctamente el cuadrante y funciona cuando la parte real es cero.

## Ejercicio 2. Potencias de $i$

Calcula manualmente $i^{15}$, $i^{26}$, $i^{51}$ e $i^{100}$. Después comprueba los resultados con Python.

In [18]:
potencias = [15, 26, 51, 100]

# TODO: calcula y muestra 1j ** exponente para cada valor

for p in potencias:
    print(1j**p)

(-0-1j)
(-1+0j)
(-0-1j)
(1+0j)


**Responde:** ¿Cada cuántas potencias se repite el patrón? ¿Cómo interviene el resto de dividir el exponente entre 4?

**Respuesta:** El patrón de las potencias de (i) se repite cada 4. El resto módulo 4 determina el resultado: 0 → (1), 1 → (i), 2 → (-1), 3 → (-i).

Por tanto: (i^{15}=-i), (i^{26}=-1), (i^{51}=-i) e (i^{100}=1).

## Ejercicio 3. Cuadrado y módulo al cuadrado

Para $z=(1+i)/2$, calcula manualmente $z^2$, $z^*$, $z^*z$, $|z|$ y $|z|^2$. Después compruébalo con NumPy.

In [19]:
z = (1 + 1j) / 2

z = np.array(z, dtype=complex)

cuadrado = z**2
conjugado = z.conj()
producto_conjugado = z.conj()*z
modulo = abs(z)
modulo_cuadrado = abs(z)**2

print("z²:", cuadrado)
print("z*:", conjugado)
print("z*z:", producto_conjugado)
print("|z|:", modulo)
print("|z|²:", modulo_cuadrado)

z²: 0.5j
z*: (0.5-0.5j)
z*z: (0.5+0j)
|z|: 0.7071067811865476
|z|²: 0.5000000000000001


**Responde:** ¿Por qué $z^2$ no puede utilizarse como probabilidad? ¿Por qué $z^*z$ siempre es real y no negativo? ¿Puede una amplitud tener módulo mayor que 1 en un estado normalizado?

**Respuesta:** 
Para (z=(1+i)/2): (z^2=i/2), (z^*=(1-i)/2), (z^*z=1/2), (|z|=1/\sqrt2) y (|z|^2=1/2).

(z^2) no sirve como probabilidad porque puede ser complejo. (z^*z) siempre es real y no negativo porque es el módulo al cuadrado. En un estado normalizado ninguna amplitud puede tener módulo mayor que 1.

## Ejercicio 4. Forma cartesiana y polar

Implementa funciones para obtener el módulo y la fase de un número complejo y para reconstruirlo a partir de su forma polar.

In [20]:
def datos_polares(z):
    """Devuelve módulo, fase en radianes y fase en grados."""
    z = complex(z)
    fase = np.angle(z)
    return np.abs(z), fase, np.degrees(fase)


def crear_complejo(modulo, fase):
    """Construye un complejo a partir de módulo y fase."""
    if not np.isfinite(modulo) or not np.isfinite(fase):
        raise ValueError("El módulo y la fase deben ser finitos")
    if modulo < 0:
        raise ValueError("El módulo no puede ser negativo")
    return modulo*np.exp(1j*fase)


numeros_polares = [1 + 1j, -1 + 1j, -1j, 2 * np.exp(1j * np.pi / 3)]

# TODO: convierte cada número a forma polar, reconstrúyelo y comprueba el resultado
for numero in numeros_polares:
    modulo,fase,grados=datos_polares(numero)
    reconstruido=crear_complejo(modulo, fase)
    print(f"z={z:.5f} -> r={modulo:.5f}, 0 = {fase:.5f} rad({grados:.2f}º)")
    print("reconstruido:", f"{reconstruido:.5f}", "correcto:", np.allclose(z, reconstruido))




z=0.50000+0.50000j -> r=1.41421, 0 = 0.78540 rad(45.00º)
reconstruido: 1.00000+1.00000j correcto: False
z=0.50000+0.50000j -> r=1.41421, 0 = 2.35619 rad(135.00º)
reconstruido: -1.00000+1.00000j correcto: False
z=0.50000+0.50000j -> r=1.00000, 0 = -1.57080 rad(-90.00º)
reconstruido: 0.00000-1.00000j correcto: False
z=0.50000+0.50000j -> r=2.00000, 0 = 1.04720 rad(60.00º)
reconstruido: 1.00000+1.73205j correcto: False


**Responde:** ¿Por qué dos fases que difieren en $2\pi$ representan el mismo número complejo?

**Respuesta:** Porque (e^{i(\theta+2\pi)}=e^{i\theta}); una vuelta completa no cambia el número complejo.

# Parte B. Notación de Dirac y producto interno

## Ejercicio 5. Del *ket* al *bra*

Representa

$$|\psi\rangle=\frac{1}{\sqrt2}\begin{pmatrix}1\\i\end{pmatrix}$$

como array unidimensional y como matriz columna. Obtén el *bra* asociado en ambos casos y muestra sus formas con `shape`.

In [21]:
# TODO: crea ambas representaciones
psi_vector = np.array([1,1j], dtype=complex) / np.sqrt(2)
print("Estado psi:", psi_vector, "norma:", np.linalg.norm(psi_vector), "forma:", psi_vector.shape)
psi_columna = psi_vector.reshape(2,1)
print("Estado psi como columna:\n", psi_columna, "norma:", np.linalg.norm(psi_columna), "forma:", psi_columna.shape)

# TODO: calcula los bras correspondientes
#transpuesta conjugada
#np.conj(psi_columna).T
bra_matriz = psi_columna.conj().T # transpuesta conjugada de la columna(daga)

# TODO: muestra valores y shapes
print("Ket como vector:", psi_vector, "shape:", psi_vector.shape)
print("Ket como columna:", psi_columna, "shape:", psi_columna.shape)
print("Bra:", bra_matriz, "shape:", bra_matriz.shape)




Estado psi: [0.70711+0.j      0.     +0.70711j] norma: 0.9999999999999999 forma: (2,)
Estado psi como columna:
 [[0.70711+0.j     ]
 [0.     +0.70711j]] norma: 0.9999999999999999 forma: (2, 1)
Ket como vector: [0.70711+0.j      0.     +0.70711j] shape: (2,)
Ket como columna: [[0.70711+0.j     ]
 [0.     +0.70711j]] shape: (2, 1)
Bra: [[0.70711-0.j      0.     -0.70711j]] shape: (1, 2)


**Responde:**

1. ¿Por qué `psi_vector.T` no cambia la forma del array?
2. ¿Qué operación debe realizarse además de la transposición?
3. ¿Qué formas tienen el *ket* columna y su *bra*?

**Respuesta:** 

-   Porque un array 1D no tiene filas y columnas que intercambiar
-   Además de transponer, hay que tomar el conjugado complejo
-   El ket columna tiene forma (2,1) y el bra (1,2)

## Ejercicio 6. Producto interno

Implementa $\langle a|b\rangle$ sin utilizar inicialmente `np.vdot()`. Recuerda conjugar el primer vector.

In [22]:
def producto_interno(estado_a, estado_b):
    """Calcula <estado_a|estado_b>."""
    a = np.asarray(estado_a, dtype=complex)
    b = np.asarray(estado_b, dtype=complex)

    if a.shape != b.shape or a.ndim != 1:
        raise ValueError("Los estados deben ser vectores unidimensionales de igual dimension")
    if not np.all(np.isfinite(a)) or not np.all(np.isfinite(b)):
        raise ValueError("Los vectores deben contener valores finitos")
    return np.sum(np.conj(a)*b)


# TODO: calcula <0|0>, <1|1>, <0|1> y <1|0>

productos ={
    "<0|0>": producto_interno(ket_0,ket_0),
    "<1|1>": producto_interno(ket_1,ket_1),
    "<0|1>": producto_interno(ket_0,ket_1),
    "<1|0>": producto_interno(ket_1,ket_0)
}

for name, producto in productos.items():
    print(f"{name}: {producto}")

    izquierda, derecha = name[1:-1].split("|")
    a= ket_0 if izquierda == "0" else ket_1
    b= ket_0 if derecha == "0" else ket_1
    print(name, "=", producto, "| coincide con np.vdot:", np.allclose(producto, np.vdot(a,b)))
# TODO: compara cada resultado con np.vdot()

<0|0>: (1+0j)
<0|0> = (1+0j) | coincide con np.vdot: True
<1|1>: (1+0j)
<1|1> = (1+0j) | coincide con np.vdot: True
<0|1>: 0j
<0|1> = 0j | coincide con np.vdot: True
<1|0>: 0j
<1|0> = 0j | coincide con np.vdot: True


## Ejercicio 7. Propiedades del producto interno

Construye

$$|\psi\rangle=\frac{|0\rangle+i|1\rangle}{\sqrt2},\qquad
|\phi\rangle=\frac{|0\rangle-|1\rangle}{\sqrt2}.$$

Comprueba que $\langle\psi|\phi\rangle=\langle\phi|\psi\rangle^*$.

In [23]:
# TODO: construye psi y phi
psi = np.array([(1+1j)/ np.sqrt(2)], dtype=complex)
phi = np.array([(1-1j)/ np.sqrt(2)], dtype=complex)

producto_psi_phi = producto_interno(psi, phi)
producto_phi_psi = producto_interno(phi, psi)

print("<psi|phi> =", producto_psi_phi)
print("<phi|psi> =", producto_phi_psi)

print("<psi|phi> =", producto_psi_phi)
print("<phi|psi>* =", producto_phi_psi.conj())
# TODO: añade un assert que compruebe la relación de conjugación

<psi|phi> = (2.2371143170757382e-17-0.9999999999999998j)
<phi|psi> = (2.2371143170757382e-17+0.9999999999999998j)
<psi|phi> = (2.2371143170757382e-17-0.9999999999999998j)
<phi|psi>* = (2.2371143170757382e-17-0.9999999999999998j)


**Responde:** ¿El producto interno debe ser real? ¿Qué propiedad se cumple al intercambiar los estados? ¿Qué error aparece si no se conjuga el primer vector?

**Respuesta:** 
El producto interno no tiene que ser real; puede ser complejo. Al intercambiar los estados se obtiene el conjugado:  <a|b><b|a>*. Si no se conjuga el primer vector, el producto interno se calcula incorrectamente y pierde sus propiedades.

# Parte C. Ortogonalidad y bases

## Ejercicio 8. Bases $X$ e $Y$

Construye los estados $|+\rangle$, $|-\rangle$, $|+i\rangle$ y $|-i\rangle$. Para cada uno muestra el vector, la norma, las probabilidades en la base computacional y su fase relativa.

In [24]:
# TODO: construye los estados
ket_plus = np.array([1/np.sqrt(2) , 1/np.sqrt(2)], dtype=complex)
ket_minus = np.array([1/np.sqrt(2) , -1/np.sqrt(2)], dtype=complex)
ket_mas_i = np.array([1/np.sqrt(2), +1j/np.sqrt(2)], dtype=complex)
ket_menos_i = np.array([1/np.sqrt(2), -1j/np.sqrt(2)], dtype=complex)

estados_bases = {
    "|+>": ket_plus,
    "|->": ket_minus,
    "|+i>": ket_mas_i,
    "|-i>": ket_menos_i,
}

# TODO: muestra norma, probabilidades y fase relativa de cada estado

for n,estado in estados_bases.items():
    norma = np.linalg.norm(estado)
    print(f"{n}: Norma: {norma}, Probabilidad: {np.abs(estado) ** 2} y la fase relativa: {np.angle(z):.5f}")


|+>: Norma: 0.9999999999999999, Probabilidad: [0.5 0.5] y la fase relativa: 0.78540
|->: Norma: 0.9999999999999999, Probabilidad: [0.5 0.5] y la fase relativa: 0.78540
|+i>: Norma: 0.9999999999999999, Probabilidad: [0.5 0.5] y la fase relativa: 0.78540
|-i>: Norma: 0.9999999999999999, Probabilidad: [0.5 0.5] y la fase relativa: 0.78540


**Responde:** ¿Producen los cuatro estados las mismas probabilidades en la base computacional? ¿Son el mismo estado? ¿Qué información los diferencia? ¿Qué pares deben ser ortogonales?

**Respuesta:** Los cuatro estados producen probabilidades (1/2) y (1/2) en la base computacional. No son el mismo estado, porque tienen distintas fases relativas. Los pares ortogonales son |+>,|-> y |+i>,|-i>.

## Ejercicio 9. Comprobación de ortogonalidad

Implementa una función que valide los estados y determine si su producto interno es aproximadamente cero.

In [25]:
def son_ortogonales(estado_a, estado_b, tolerancia=1e-10):
    """Devuelve True si dos estados válidos son ortogonales."""
    producto_interno = np.vdot(estado_a, estado_b)
    return np.abs(producto_interno) < tolerancia

pares = [
    ("|0>, |1>", ket_0, ket_1),
    ("|+>, |->", ket_plus, ket_minus),
    ("|+i>, |-i>", ket_mas_i, ket_menos_i),
    ("|0>, |+>", ket_0, ket_plus),
    ("|1>, |->", ket_1, ket_minus),
    ("|+>, |+i>", ket_plus, ket_mas_i),
]


for nombre, estado_a, estado_b in pares:

    print(estado_a, " ", estado_b)
    producto_interno = np.vdot(estado_a, estado_b)
    modulo = np.abs(producto_interno)

    print(f"{nombre}")
    print(f"Producto interno: {producto_interno}")
    print(f"Módulo: {modulo}")
    print(f"¿Son ortogonales?: {son_ortogonales(estado_a, estado_b)}")
    print()

# TODO: muestra para cada par el producto interno, su módulo y si son ortogonales

[1.+0.j 0.+0.j]   [0.+0.j 1.+0.j]
|0>, |1>
Producto interno: 0j
Módulo: 0.0
¿Son ortogonales?: True

[0.70711+0.j 0.70711+0.j]   [ 0.70711+0.j -0.70711+0.j]
|+>, |->
Producto interno: (-2.2371143170757382e-17+0j)
Módulo: 2.2371143170757382e-17
¿Son ortogonales?: True

[0.70711+0.j      0.     +0.70711j]   [ 0.70711+0.j      -0.     -0.70711j]
|+i>, |-i>
Producto interno: 0j
Módulo: 0.0
¿Son ortogonales?: True

[1.+0.j 0.+0.j]   [0.70711+0.j 0.70711+0.j]
|0>, |+>
Producto interno: (0.7071067811865475+0j)
Módulo: 0.7071067811865475
¿Son ortogonales?: False

[0.+0.j 1.+0.j]   [ 0.70711+0.j -0.70711+0.j]
|1>, |->
Producto interno: (-0.7071067811865475+0j)
Módulo: 0.7071067811865475
¿Son ortogonales?: False

[0.70711+0.j 0.70711+0.j]   [0.70711+0.j      0.     +0.70711j]
|+>, |+i>
Producto interno: (0.4999999999999999+0.4999999999999999j)
Módulo: 0.7071067811865475
¿Son ortogonales?: False



## Ejercicio 10. Base ortonormal

Implementa una función que compruebe que dos estados están normalizados y son ortogonales.

In [30]:
def es_base_ortonormal(estado_a, estado_b, tolerancia=1e-10):
    """Determina si dos estados forman una base ortonormal de un qubit."""

    a_normalizado = np.isclose(np.linalg.norm(estado_a), 1, atol=tolerancia)
    b_normalizado = np.isclose(np.linalg.norm(estado_b), 1, atol=tolerancia)
    ortogonales = np.abs(np.vdot(estado_a, estado_b)) < tolerancia

    return a_normalizado and b_normalizado and ortogonales


bases = {
    "base_Z": (ket_0, ket_1),
    "base_X": (ket_plus, ket_minus),
    "base_Y": (ket_mas_i, ket_menos_i),
    "base_repetida": (ket_0, ket_0),
    "no_ortogonal": (ket_0, ket_plus),
    "no_normalizada": (np.array([1, 1], dtype=complex), ket_1),
}


for nombre, (estado_a, estado_b) in bases.items():
    norma_a = np.linalg.norm(estado_a)
    norma_b = np.linalg.norm(estado_b)
    producto_interno = np.vdot(estado_a, estado_b)

    print(nombre)
    print("  Norma estado A:", norma_a)
    print("  Norma estado B:", norma_b)
    print("  Producto interno:", producto_interno)
    print("  ¿Es base ortonormal?:",
          es_base_ortonormal(estado_a, estado_b))
    print()

base_Z
  Norma estado A: 1.0
  Norma estado B: 1.0
  Producto interno: 0j
  ¿Es base ortonormal?: True

base_X
  Norma estado A: 0.9999999999999999
  Norma estado B: 0.9999999999999999
  Producto interno: (-2.2371143170757382e-17+0j)
  ¿Es base ortonormal?: True

base_Y
  Norma estado A: 0.9999999999999999
  Norma estado B: 0.9999999999999999
  Producto interno: 0j
  ¿Es base ortonormal?: True

base_repetida
  Norma estado A: 1.0
  Norma estado B: 1.0
  Producto interno: (1+0j)
  ¿Es base ortonormal?: False

no_ortogonal
  Norma estado A: 1.0
  Norma estado B: 0.9999999999999999
  Producto interno: (0.7071067811865475+0j)
  ¿Es base ortonormal?: False

no_normalizada
  Norma estado A: 1.4142135623730951
  Norma estado B: 1.0
  Producto interno: (1+0j)
  ¿Es base ortonormal?: False



## Ejercicio 11. Cambio de base

Los coeficientes de $|\psi\rangle$ en una base ortonormal $\{|u\rangle,|v\rangle\}$ son $\langle u|\psi\rangle$ y $\langle v|\psi\rangle$.

In [ ]:
def coordenadas_en_base(estado, base):
    """Calcula las coordenadas de un estado en una base ortonormal."""
    return np.array([np.vdot(vector_base, estado) for vector_base in base])


base_Z = [ket_0, ket_1]
base_X = [ket_plus, ket_minus]
base_Y = [ket_mas_i, ket_menos_i]


casos = [
    ("|0> en base X", ket_0, base_X),
    ("|1> en base X", ket_1, base_X),
    ("|+> en base Z", ket_plus, base_Z),
    ("|-i> en base Y", ket_menos_i, base_Y),
]


for nombre, estado, base in casos:
    coordenadas = coordenadas_en_base(estado, base)

    # Reconstrucción:
    reconstruido = sum(
        coef * vector_base
        for coef, vector_base in zip(coordenadas, base)
    )

    print(nombre)
    print("  Coordenadas:", coordenadas)
    print("  Estado original:", estado)
    print("  Reconstruido:", reconstruido)
    print("  ¿Coinciden?:", np.allclose(estado, reconstruido))
    print()

**Responde:** ¿Cambiar de base modifica el estado físico? ¿Qué cambia? ¿Qué coordenadas debe tener $|-i\rangle$ en la base $Y$?

**Respuesta:** No. Cambiar de base no modifica el estado físico; solo cambia la forma de representarlo matemáticamente. Lo que cambia son sus coordenadas respecto a la base elegida. Es parecido a describir el mismo vector usando ejes distintos: el vector es el mismo, pero sus componentes cambian.

# Parte D. Fase global y fase relativa

## Ejercicio 12. Comparación de estados

Considera:

$$|\psi_A\rangle=\frac{1}{\sqrt2}(1,i)^T,\quad
|\psi_B\rangle=\frac{1}{\sqrt2}(i,-1)^T,$$

$$|\psi_C\rangle=\frac{1}{\sqrt2}(1,-i)^T,\quad
|\psi_D\rangle=-\frac{1}{\sqrt2}(1,i)^T.$$

Comprueba sus normas y probabilidades. Calcula el producto interno de $A$ con cada estado y determina cuáles son físicamente equivalentes a $A$.

In [31]:

# Estados
psi_A = np.array([1, 1j], dtype=complex) / np.sqrt(2)
psi_B = np.array([1j, -1], dtype=complex) / np.sqrt(2)
psi_C = np.array([1, -1j], dtype=complex) / np.sqrt(2)
psi_D = -np.array([1, 1j], dtype=complex) / np.sqrt(2)

estados = {
    "A": psi_A,
    "B": psi_B,
    "C": psi_C,
    "D": psi_D
}

for nombre, psi in estados.items():
    norma = np.linalg.norm(psi)
    probabilidades = np.abs(psi)**2
    producto = np.vdot(psi_A, psi)

    print(f"psi_{nombre}")
    print("  Norma:", norma)
    print("  Probabilidades:", probabilidades)
    print("  <A|psi> =", producto)
    print("  |<A|psi>| =", np.abs(producto))
    print()

psi_A
  Norma: 0.9999999999999999
  Probabilidades: [0.5 0.5]
  <A|psi> = (0.9999999999999998+0j)
  |<A|psi>| = 0.9999999999999998

psi_B
  Norma: 0.9999999999999999
  Probabilidades: [0.5 0.5]
  <A|psi> = 0.9999999999999998j
  |<A|psi>| = 0.9999999999999998

psi_C
  Norma: 0.9999999999999999
  Probabilidades: [0.5 0.5]
  <A|psi> = 0j
  |<A|psi>| = 0.0

psi_D
  Norma: 0.9999999999999999
  Probabilidades: [0.5 0.5]
  <A|psi> = (-0.9999999999999998+0j)
  |<A|psi>| = 0.9999999999999998



**Responde:** ¿Qué factor relaciona $A$ con $B$? ¿Y $A$ con $D$? ¿Por qué $A$ y $C$ no son equivalentes aunque produzcan las mismas probabilidades?

**Respuesta:** A con B se relaciona por el factor i.
A con D se relaciona por el factor -1.
A y C no son equivalentes porque, aunque dan las mismas probabilidades al medir, tienen distinta fase relativa entre sus componentes.

## Ejercicio 13. Equivalencia salvo fase global

Para estados normalizados, utiliza el criterio $|\langle\psi|\phi\rangle|=1$. No dividas componentes, porque alguna podría ser cero.

In [32]:
def equivalentes_fase_global(estado_a, estado_b, tolerancia=1e-10):
    """Determina si dos estados válidos difieren solo en una fase global."""
    producto_interno = np.vdot(estado_a, estado_b)
    return np.isclose(np.abs(producto_interno), 1, atol=tolerancia)


# Pruebas
print("A y B:", equivalentes_fase_global(psi_A, psi_B))  # True
print("A y D:", equivalentes_fase_global(psi_A, psi_D))  # True
print("A y C:", equivalentes_fase_global(psi_A, psi_C))  # False
print("|0> y |1>:", equivalentes_fase_global(ket_0, ket_1))  # False

A y B: True
A y D: True
A y C: False
|0> y |1>: False


**Responde:** ¿Por qué es problemático dividir una componente entre otra? ¿Por qué el módulo del producto interno elimina la fase global? ¿Puede aplicarse directamente el criterio a vectores no normalizados?

**Respuesta:** 

¿Por qué no dividir componentes? Porque alguna componente puede ser cero y producir una división por cero.

¿Por qué el módulo elimina la fase global? Porque una fase global tiene módulo 1, así que al tomar el valor absoluto desaparece su efecto.

¿Sirve directamente para vectores no normalizados? No. El criterio supone que ambos vectores tienen norma 1. Si no están normalizados, primero hay que normalizarlos.